In [1]:
import websocket
import json
import csv
import datetime
import os
import pandas as pd
import threading
import time
from collections import deque
from collections import defaultdict

In [2]:
WEBSOCKET_URL =  "wss://ws.bitget.com/v2/ws/public"
INSTRUMENT_IDS = ["SOLUSDT", "BTCUSDT", "ETHUSDT"]  
TRADE_CSV_FILE = "trade_data.csv"

In [3]:
def tao_file_csv():
    header = [
    "thoi_gian",          # Thời gian hệ thống ghi nhận
    "timestamp_api",      # Timestamp từ API (ts)
    "instId",             # Mã sản phẩm (ví dụ: BTCUSDT)
    "trade_id",           # ID giao dịch
     "price",              # Giá giao dịch
    "size",               # Khối lượng giao dịch
    "side",               # Hướng giao dịch (buy/sell)
    "action"              # snapshot (loại push)
]


    if not os.path.exists(TRADE_CSV_FILE) or os.path.getsize(TRADE_CSV_FILE) == 0:
        with open(TRADE_CSV_FILE, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(header)
        print(f"Đã tạo file CSV với {len(header)} cột: {TRADE_CSV_FILE}")
    else:
        print(f"File CSV đã tồn tại: {TRADE_CSV_FILE}")

tao_file_csv()

Đã tạo file CSV với 8 cột: trade_data.csv


In [ ]:
def on_open(ws):
    print(f" Đã kết nối thành công")
    reset_thread = threading.Thread(target=reset_counters, daemon=True)
    reset_thread.start()

    for inst_id in INSTRUMENT_IDS:
        subscribe_message = {
            "op": "subscribe",
            "args": [
                {
                    "instType": "SPOT",
                    "channel": "trade",
                    "instId": inst_id
                }
            ]
        }
        ws.send(json.dumps(subscribe_message))
        print(f"Đang theo dõi {inst_id}")
    print(f"Đang theo dõi các cặp: {', '.join(INSTRUMENT_IDS)}")

all_data= []
trade_counters = defaultdict(int)
MAX_TRADES_PER_SYMBOL = 50
def reset_counters():
    while True:
        time.sleep(60)  
        global trade_counters
        old_counters = dict(trade_counters)
        trade_counters.clear()
        print(f" Reset counters sau 1 phút. Trades đã lưu: {old_counters}")

def on_message(ws, message_str):  
    global all_data, trade_counters
    data = json.loads(message_str)
    all_data.append(data)
    
    
    print(f"Received: {data}")
    
    
    if "event" in data:
        print(f"Subscription event: {data.get('event')} for {data.get('arg', {}).get('instId')}")
        return
    
    if "data" in data and data["data"]:
        instId = data.get("arg", {}).get("instId")
        action = data.get('action', 'unknown')
        
        # Bỏ qua snapshot data để tránh spam
        # if action == 'snapshot':
        #     print(f" Bỏ qua snapshot data cho {instId}")
        #     return
        
        
        if trade_counters[instId] >= MAX_TRADES_PER_SYMBOL:
            print(f" {instId} đã đạt giới hạn {MAX_TRADES_PER_SYMBOL} trades trong phút này")
            return
        
        for trade in data["data"]:
           
            if trade_counters[instId] >= MAX_TRADES_PER_SYMBOL:
                print(f" {instId} đã đạt giới hạn {MAX_TRADES_PER_SYMBOL} trades")
                break
                
            thoi_gian = datetime.datetime.now().isoformat()
            timestamp_api = trade.get('ts')
            trade_id = trade.get('tradeId')
            price = trade.get('price')
            size = trade.get('size')
            side = trade.get('side')
            
            
            trade_counters[instId] += 1
            
            
            print(f" {instId} [{trade_counters[instId]}/{MAX_TRADES_PER_SYMBOL}] | Giá: {price} | Size: {size} | Side: {side}")
            
            # Lưu vào CSV
            with open(TRADE_CSV_FILE, mode='a', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow([thoi_gian, timestamp_api, instId, trade_id, price, size, side, action])
    
    
    else:
        pass 

def on_error(ws, error):
    print(f" Lỗi: {error}")

def on_close(ws, close_status_code, close_msg):
    print(f" Kết nối đã đóng - Code: {close_status_code}")
    # In thống kê cuối cùng
    print(" Thống kê cuối cùng:")
    for symbol, count in trade_counters.items():
        print(f" {symbol}: {count} trades")

In [ ]:
a=60  #1 phút
b=a*60  #1h
c=b*24  #1 ngày
def run_ws():
    ws.run_forever(ping_interval=30, ping_timeout=10)
ws = websocket.WebSocketApp(WEBSOCKET_URL,
                          on_open=on_open,
                          on_message=on_message,
                          on_error=on_error,
                          on_close=on_close)

print(f"Bắt đầu kết nối đến Bitget...")
print(f"Dữ liệu sẽ được lưu vào: {TRADE_CSV_FILE}")
print("Nhấn Ctrl+C để dừng")

ws_thread = threading.Thread(target=run_ws)
ws_thread.daemon = True
ws_thread.start()

run_duration = a 

try:
    time.sleep(run_duration)
except KeyboardInterrupt:
    print("\nĐã dừng bằng Ctrl+C")


ws.close()
print(f"Đã ngắt kết nối sau {run_duration} giây.")

Bắt đầu kết nối đến Bitget...
Dữ liệu sẽ được lưu vào: trade_data.csv
Nhấn Ctrl+C để dừng
 Đã kết nối thành công
Đang theo dõi SOLUSDT
Đang theo dõi BTCUSDT
Đang theo dõi ETHUSDT
Đang theo dõi các cặp: SOLUSDT, BTCUSDT, ETHUSDT
Received: {'event': 'subscribe', 'arg': {'instType': 'SPOT', 'channel': 'trade', 'instId': 'SOLUSDT'}}
Subscription event: subscribe for SOLUSDT
Received: {'event': 'subscribe', 'arg': {'instType': 'SPOT', 'channel': 'trade', 'instId': 'BTCUSDT'}}
Subscription event: subscribe for BTCUSDT
Received: {'event': 'subscribe', 'arg': {'instType': 'SPOT', 'channel': 'trade', 'instId': 'ETHUSDT'}}
Subscription event: subscribe for ETHUSDT
Received: {'action': 'snapshot', 'arg': {'instType': 'SPOT', 'channel': 'trade', 'instId': 'SOLUSDT'}, 'data': [{'ts': '1749098817974', 'price': '153.99', 'size': '12.2380', 'side': 'buy', 'tradeId': '1314381196624863233'}, {'ts': '1749098817971', 'price': '153.99', 'size': '0.5870', 'side': 'buy', 'tradeId': '1314381196612280327'}, {'

 Reset counters sau 1 phút. Trades đã lưu: {'SOLUSDT': 50, 'BTCUSDT': 50, 'ETHUSDT': 50}
 Reset counters sau 1 phút. Trades đã lưu: {}
